In [16]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import StandardScaler
import ssl
import pandas as pd

In [4]:
df = pd.read_csv("GoEmotions/data/train.tsv", sep="\t", header=None)
df.to_csv("GoEmotions_train.csv", index=False)

df = pd.read_csv("GoEmotions/data/test.tsv", sep="\t", header=None)
df.to_csv("GoEmotions_test.csv", index=False)

with open("GoEmotions/data/emotions.txt") as f:
    emotions = [line.strip() for line in f]

df = pd.DataFrame({"emotion": emotions})
df.to_csv("emotions.csv", index=False)

In [5]:
import os

raw_dir = "SemEvalDataset"
out_dir = "SemEvalDataset/processed"

os.makedirs(out_dir, exist_ok=True)

files = {
    "train": "2018-E-c-En-train.txt",
    "dev": "2018-E-c-En-dev.txt",
    "test": "2018-E-c-En-test.txt"
}

for split, filename in files.items():
    path = os.path.join(raw_dir, filename)

    df = pd.read_csv(path, sep="\t")

    output_path = os.path.join(out_dir, f"{split}.csv")
    df.to_csv(output_path, index=False)

    print(f"Saved {output_path}")

Saved SemEvalDataset/processed/train.csv
Saved SemEvalDataset/processed/dev.csv
Saved SemEvalDataset/processed/test.csv


# Loading and Preprocessing:

In [17]:
G_train = pd.read_csv("GoEmotions/Processed/GoEmotions_train.csv")
G_test = pd.read_csv("GoEmotions/Processed/GoEmotions_test.csv")

S_train = pd.read_csv("SemEvalDataset/processed/train.csv")
S_test = pd.read_csv("SemEvalDataset/processed/test.csv")

T_train = pd.read_csv("TwitterEmotions/training.csv")
T_test = pd.read_csv("TwitterEmotions/test.csv")

In [18]:
ssl._create_default_https_context = ssl._create_unverified_context

nltk.download('stopwords')
nltk.download('punkt')
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/varunpanuganti/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/varunpanuganti/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [19]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_text(text):
    return word_tokenize(clean_text(text))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

def pad_tokens(tokens, max_len=100, padding="post"):
    tokens = tokens[:max_len]

    pad_amount = max_len - len(tokens)

    if padding == "post":
        return tokens + ["<PAD>"] * pad_amount
    else:
        return ["<PAD>"] * pad_amount + tokens

In [20]:
def preprocess_only(df, text_col, max_len=100):
    df = df.copy()

    df["clean_text"] = df[text_col].apply(clean_text)

    df["tokens"] = df[text_col].apply(tokenize_text)

    df["tokens_no_stopwords"] = df["tokens"].apply(remove_stopwords)

    df["padded_tokens"] = df["tokens_no_stopwords"].apply(
        lambda tokens: pad_tokens(tokens, max_len=max_len, padding="post")
    )

    df["word_count"] = df["tokens"].apply(len)
    df["clean_word_count"] = df["tokens_no_stopwords"].apply(len)
    df["char_count"] = df["clean_text"].apply(len)

    scaler = StandardScaler()
    df[["word_count_scaled", "clean_word_count_scaled", "char_count_scaled"]] = scaler.fit_transform(
        df[["word_count", "clean_word_count", "char_count"]]
    )

    return df

In [21]:
twitter_train = preprocess_only(T_train, text_col="text", max_len=100)
twitter_test = preprocess_only(T_test, text_col="text", max_len=100)

semeval_train = preprocess_only(S_train, text_col="Tweet", max_len=100)
semeval_test = preprocess_only(S_test, text_col="Tweet", max_len=100)

goemotions_train = preprocess_only(G_train, text_col="text", max_len=100)
goemotions_test = preprocess_only(G_test, text_col="text", max_len=100)

LookupError: 
**********************************************************************
  Resource 'punkt_tab' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'tokenizers/punkt_tab/english/'

  Searched in:
    - '/Users/varunpanuganti/nltk_data'
    - '/Users/varunpanuganti/Desktop/NSF-REU-Project/venv/nltk_data'
    - '/Users/varunpanuganti/Desktop/NSF-REU-Project/venv/share/nltk_data'
    - '/Users/varunpanuganti/Desktop/NSF-REU-Project/venv/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************
